# Interopérabilité avec C : le module `ctypes`

Cette démonstration écrit deux fonctions C, le produit et la somme cumulée des valeurs d'un tableau, les compile en bibliothèque partagée, puis les appelle depuis Python avec `ctypes`. Elle compare enfin le temps de la somme cumulée en C à celui d'une boucle Python et de `np.cumsum`, comme le fait la démonstration Cython.

## Création du fichier C

In [1]:
%%writefile functions.c
double prod(double *x, int n) {
  double prod = 1.0;
  for (int i = 0; i < n; i++) {
    prod *= x[i];
  }
  return prod;
}

void cumsum(double *a, double *b, int n) {
  if (n <= 0) {
    return;
  }
  b[0] = a[0];
  for (int i = 1; i < n; i++) {
    b[i] = a[i] + b[i - 1];
  }
}

Writing functions.c


## Compilation de la bibliothèque partagée

In [2]:
!gcc -c -Wall -O2 -fPIC functions.c -o functions.o
!gcc -shared -o libfunctions.so functions.o

## Chargement de la bibliothèque et déclaration des types des fonctions

In [3]:
import ctypes
import pathlib

import numpy as np

libname = "./libfunctions.so"
if not pathlib.Path(libname).exists():
  raise FileNotFoundError(f"{libname} not found")

lib = ctypes.CDLL(libname)

lib.prod.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_int]
lib.prod.restype = ctypes.c_double

lib.cumsum.argtypes = [
  ctypes.POINTER(ctypes.c_double),
  ctypes.POINTER(ctypes.c_double),
  ctypes.c_int,
]
lib.cumsum.restype = None

## Des fonctions Python autour des fonctions C

Elles convertissent leur argument en tableau NumPy de `float64` contigu, le type qu'attend le C, et passent un pointeur vers ses données.

In [4]:
def prod(x):
  arr = np.ascontiguousarray(x, dtype=np.float64)
  ptr = arr.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  return lib.prod(ptr, len(arr))


def cumsum(a):
  a = np.ascontiguousarray(a, dtype=np.float64)
  b = np.empty_like(a)
  ptr_a = a.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  ptr_b = b.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  lib.cumsum(ptr_a, ptr_b, len(a))
  return b

## Utilisation des fonctions en Python

In [5]:
print(prod([1, 2, 3, 4]))
print(cumsum([1, 2, 3, 4]))
print(cumsum([]))

24.0
[ 1.  3.  6. 10.]
[]


## Mesure du temps d'exécution

La somme cumulée d'un million de valeurs, de trois façons : une boucle Python, sur une liste, puis la fonction C, puis `np.cumsum`.

In [6]:
def cumsum_python(valeurs):
  b = []
  total = 0.0
  for x in valeurs:
    total += x
    b.append(total)
  return b


a = np.linspace(-1, 1, 1_000_000)
valeurs = a.tolist()

In [7]:
%timeit cumsum_python(valeurs)

81.1 ms ± 12.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [8]:
%timeit cumsum(a)

2.47 ms ± 184 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [9]:
%timeit np.cumsum(a)

5.06 ms ± 127 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


La fonction C est plus de dix fois plus rapide que la boucle Python, et au moins aussi rapide que `np.cumsum`, elle-même écrite en C.